# Fusion: ResNet18 + CatBoost

## 1. Библиотеки

In [ ]:
%pip install -q catboost iterative-stratification


In [ ]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18
from PIL import Image
from tqdm.auto import tqdm
from catboost import CatBoostClassifier
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import f1_score

DATA_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data' / 'train.csv').is_file())
ROOT = DATA_ROOT / 'archive'
TRAIN_DIR = DATA_ROOT / 'data' / 'train'
ARTIFACTS_DIR = ROOT / 'artifacts'

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


## 2. Данные и split

In [ ]:
train = pd.read_csv(DATA_ROOT / 'data' / 'train.csv')
target_columns = train.columns.drop('item_id').tolist()

splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=42
)

train_val_index, test_index = next(
    splitter.split(train, train[target_columns])
)

train_val_df = train.iloc[train_val_index].reset_index(drop=True)
test_df = train.iloc[test_index].reset_index(drop=True)

splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.1765,
    random_state=43
)

train_index, val_index = next(
    splitter.split(
        train_val_df,
        train_val_df[target_columns]
    )
)

val_df = train_val_df.iloc[val_index].reset_index(drop=True)
val_df['image_path'] = val_df['item_id'].apply(
    lambda item_id: TRAIN_DIR / f'{item_id}.png'
)

print('Validation:', len(val_df))
print('Internal test:', len(test_df))
print('Missing images:', (~val_df['image_path'].apply(Path.exists)).sum())


## 3. CatBoost после подбора параметров

In [ ]:
with open(ARTIFACTS_DIR / 'catboost_tuned_config.json', encoding='utf-8') as file:
    catboost_config = json.load(file)

assert catboost_config['target_columns'] == target_columns
feature_columns = catboost_config['feature_columns']

mesh_features = pd.read_csv(
    ARTIFACTS_DIR / 'train_mesh_features.csv'
).set_index('item_id')
X_val = mesh_features.loc[val_df['item_id'], feature_columns]
X_val = X_val.replace([np.inf, -np.inf], np.nan).fillna(0)

assert list(X_val.index) == val_df['item_id'].tolist()

catboost_probabilities = np.zeros((len(val_df), len(target_columns)))

for index, target in enumerate(target_columns):
    model = CatBoostClassifier()
    model.load_model(str(ROOT / 'models' / 'catboost_tuned' / f'{target}.cbm'))
    catboost_probabilities[:, index] = model.predict_proba(X_val)[:, 1]

print('CatBoost:', catboost_config['variant'])
print('Probabilities:', catboost_probabilities.shape)


## 4. Improved ResNet18

In [ ]:
image_transform = transforms.Compose([
    transforms.Resize((224, 336)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class RenderDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        image_path = self.dataframe.loc[index, 'image_path']
        image = Image.open(image_path).convert('RGB')
        return image_transform(image)


resnet_model = resnet18(weights=None)
resnet_model.fc = nn.Linear(
    resnet_model.fc.in_features,
    len(target_columns)
)

checkpoint = torch.load(
    ROOT / 'models' / 'resnet' / 'best_resnet18_improved.pth',
    map_location=device,
    weights_only=False
)
assert checkpoint['target_columns'] == target_columns
resnet_model.load_state_dict(checkpoint['model_state_dict'])
resnet_model = resnet_model.to(device)
resnet_model.eval()

val_dataset = RenderDataset(val_df)
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


In [ ]:
resnet_probabilities = []

with torch.no_grad():
    for images in tqdm(val_loader, desc='ResNet18 validation'):
        images = images.to(device)
        logits = resnet_model(images)
        resnet_probabilities.append(torch.sigmoid(logits).cpu().numpy())

resnet_probabilities = np.concatenate(resnet_probabilities)

print('ResNet probabilities:', resnet_probabilities.shape)


## 5. Fusion

In [ ]:
val_targets = val_df[target_columns].to_numpy()


def calculate_metrics(targets, probabilities, thresholds):
    predictions = (probabilities >= thresholds).astype(int)
    defect_indices = [i for i, name in enumerate(target_columns) if name != 'quality']

    class_f1 = f1_score(
        targets,
        predictions,
        average=None,
        zero_division=0
    )
    weighted_f1 = f1_score(
        targets[:, defect_indices],
        predictions[:, defect_indices],
        average='weighted',
        zero_division=0
    )
    quality_f1 = class_f1[target_columns.index('quality')]

    return {
        'macro_f1': class_f1.mean(),
        'weighted_f1': weighted_f1,
        'quality_f1': quality_f1,
        'score': (weighted_f1 + quality_f1) / 2
    }


def find_best_thresholds(probabilities):
    threshold_grid = np.arange(0.05, 0.96, 0.01)
    thresholds = []

    for column_index in range(len(target_columns)):
        scores = [
            f1_score(
                val_targets[:, column_index],
                probabilities[:, column_index] >= threshold,
                zero_division=0
            )
            for threshold in threshold_grid
        ]
        thresholds.append(threshold_grid[np.argmax(scores)])

    return np.array(thresholds)


In [ ]:
weight_grid = np.arange(0, 1.01, 0.05)
fusion_rows = []

for resnet_weight in weight_grid:
    probabilities = (
        resnet_weight * resnet_probabilities
        + (1 - resnet_weight) * catboost_probabilities
    )
    thresholds = find_best_thresholds(probabilities)
    metrics = calculate_metrics(
        val_targets,
        probabilities,
        thresholds
    )

    fusion_rows.append({
        'resnet_weight': resnet_weight,
        **metrics
    })

fusion_results = pd.DataFrame(fusion_rows)
fusion_results.round(4)


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(
    fusion_results['resnet_weight'],
    fusion_results['score'],
    marker='o',
    color='#21A038'
)
plt.xlabel('Вес ResNet18')
plt.ylabel('Validation score')
plt.title('ResNet18 + CatBoost fusion')
plt.grid(alpha=0.3)
plt.show()


In [ ]:
best_row = fusion_results.loc[
    fusion_results['score'].idxmax()
]
best_resnet_weight = best_row['resnet_weight']

best_fusion_probabilities = (
    best_resnet_weight * resnet_probabilities
    + (1 - best_resnet_weight) * catboost_probabilities
)
best_fusion_thresholds = find_best_thresholds(
    best_fusion_probabilities
)
best_fusion_metrics = calculate_metrics(
    val_targets,
    best_fusion_probabilities,
    best_fusion_thresholds
)

resnet_metrics = calculate_metrics(
    val_targets,
    resnet_probabilities,
    find_best_thresholds(resnet_probabilities)
)
catboost_metrics = calculate_metrics(
    val_targets,
    catboost_probabilities,
    find_best_thresholds(catboost_probabilities)
)

comparison = pd.DataFrame({
    'variant': [
        'Improved ResNet18',
        'CatBoost',
        'ResNet18 + CatBoost'
    ],
    'macro_f1': [
        resnet_metrics['macro_f1'],
        catboost_metrics['macro_f1'],
        best_fusion_metrics['macro_f1']
    ],
    'weighted_f1': [
        resnet_metrics['weighted_f1'],
        catboost_metrics['weighted_f1'],
        best_fusion_metrics['weighted_f1']
    ],
    'quality_f1': [
        resnet_metrics['quality_f1'],
        catboost_metrics['quality_f1'],
        best_fusion_metrics['quality_f1']
    ],
    'score': [
        resnet_metrics['score'],
        catboost_metrics['score'],
        best_fusion_metrics['score']
    ]
})

print('Лучший вес ResNet18:', round(best_resnet_weight, 2))
display(comparison.round(4))


In [ ]:
FUSION_THRESHOLDS_PATH = (
    ARTIFACTS_DIR / 'resnet_catboost_thresholds.json'
)
FUSION_CONFIG_PATH = (
    ARTIFACTS_DIR / 'resnet_catboost_config.json'
)

thresholds_to_save = {
    target: float(threshold)
    for target, threshold in zip(
        target_columns,
        best_fusion_thresholds
    )
}

with open(FUSION_THRESHOLDS_PATH, 'w', encoding='utf-8') as file:
    json.dump(thresholds_to_save, file, indent=2)

with open(FUSION_CONFIG_PATH, 'w', encoding='utf-8') as file:
    json.dump({
        'resnet_weight': float(best_resnet_weight),
        'catboost_weight': float(1 - best_resnet_weight)
    }, file, indent=2)

print('Сохранены thresholds:', FUSION_THRESHOLDS_PATH)
print('Сохранены веса:', FUSION_CONFIG_PATH)


## 6. Fusion с новыми mesh-признаками

In [ ]:
with open(ARTIFACTS_DIR / 'catboost_features_config.json', encoding='utf-8') as file:
    feature_config = json.load(file)
assert feature_config['target_columns'] == target_columns
assert checkpoint['target_columns'] == target_columns

relative_features = pd.read_csv(
    ARTIFACTS_DIR / 'train_mesh_features_relative.csv'
).set_index('item_id')
assert relative_features.index.is_unique
X_val_relative = relative_features.loc[
    val_df['item_id'], feature_config['feature_columns']
]
assert X_val_relative.index.tolist() == val_df['item_id'].tolist()
assert np.isfinite(X_val_relative.to_numpy()).all()

feature_probabilities = np.zeros_like(catboost_probabilities)
for index, target in enumerate(target_columns):
    model = CatBoostClassifier()
    model.load_model(str(ROOT / 'models' / 'catboost_features' / f'{target}.cbm'))
    assert model.feature_names_ == feature_config['feature_columns']
    feature_probabilities[:, index] = model.predict_proba(X_val_relative)[:, 1]

print('Новый CatBoost:', feature_config['variant'])


In [ ]:
feature_fusion_rows = []
for resnet_weight in weight_grid:
    probabilities = (
        resnet_weight * resnet_probabilities
        + (1 - resnet_weight) * feature_probabilities
    )
    thresholds = find_best_thresholds(probabilities)
    feature_fusion_rows.append({
        'resnet_weight': resnet_weight,
        **calculate_metrics(val_targets, probabilities, thresholds)
    })

feature_fusion_results = pd.DataFrame(feature_fusion_rows)
feature_best_row = feature_fusion_results.loc[
    feature_fusion_results['score'].idxmax()
]
feature_resnet_weight = float(feature_best_row['resnet_weight'])
feature_fusion_probabilities = (
    feature_resnet_weight * resnet_probabilities
    + (1 - feature_resnet_weight) * feature_probabilities
)
feature_fusion_thresholds = find_best_thresholds(feature_fusion_probabilities)
feature_fusion_metrics = calculate_metrics(
    val_targets, feature_fusion_probabilities, feature_fusion_thresholds
)

display(pd.DataFrame([
    {'variant': 'Previous fusion', **best_fusion_metrics},
    {'variant': 'Fusion with relative features', **feature_fusion_metrics}
]).round(4))
print('Новый вес ResNet18:', round(feature_resnet_weight, 2))
print('Изменение score:', round(
    feature_fusion_metrics['score'] - best_fusion_metrics['score'], 4
))


In [ ]:
old_predictions = best_fusion_probabilities >= best_fusion_thresholds
new_predictions = feature_fusion_probabilities >= feature_fusion_thresholds
old_f1 = f1_score(val_targets, old_predictions, average=None, zero_division=0)
new_f1 = f1_score(val_targets, new_predictions, average=None, zero_division=0)

pd.DataFrame({
    'target': target_columns,
    'before': old_f1,
    'after': new_f1,
    'difference': new_f1 - old_f1
}).round(4)


## 7. Сохранение лучшего варианта

In [ ]:
use_new_features = feature_fusion_metrics['score'] > best_fusion_metrics['score']
selected_weight = feature_resnet_weight if use_new_features else float(best_resnet_weight)
selected_thresholds = (
    feature_fusion_thresholds if use_new_features else best_fusion_thresholds
)
selected_metrics = feature_fusion_metrics if use_new_features else best_fusion_metrics

selected_config = {
    'resnet_model': 'models/resnet/best_resnet18_improved.pth',
    'catboost_models': 'models/catboost_features' if use_new_features else 'models/catboost_tuned',
    'catboost_config': (
        'artifacts/catboost_features_config.json' if use_new_features
        else 'artifacts/catboost_tuned_config.json'
    ),
    'features_path': (
        'artifacts/train_mesh_features_relative.csv' if use_new_features
        else 'artifacts/train_mesh_features.csv'
    ),
    'target_columns': target_columns,
    'resnet_weight': selected_weight,
    'catboost_weight': 1 - selected_weight,
    'thresholds': dict(zip(target_columns, selected_thresholds.tolist())),
    'validation_score': float(selected_metrics['score']),
    'seed': SEED
}
with open(ARTIFACTS_DIR / 'resnet_catboost_features_config.json', 'w', encoding='utf-8') as file:
    json.dump(selected_config, file, indent=2, ensure_ascii=False)

print('Выбран CatBoost:', selected_config['catboost_models'])
print('Validation score:', round(selected_metrics['score'], 4))
